<a href="https://colab.research.google.com/github/Mohsin-22/Assignment6-100_Gen-Ai_cohort-/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 6: Retrieval-Augmented Generation (RAG) - Machine Learning Knowledge Assistant

In [17]:
!pip install faiss-cpu chromadb tiktoken sentence-transformers transformers
!pip install pypdf


In [18]:
from pypdf import PdfReader

reader = PdfReader("/content/intro-to-ml.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

print("Total Pages:", len(reader.pages))
print("Sample Text:", text[:1000])

Total Pages: 392
Sample Text: Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


Andreas C. Müller and Sarah Guido
Introduction to Machine Learning
with Python
A Guide for Data Scientists
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing
978-1-449-36941-5
[LSI]
Introduction to Machine Learning with Python
by Andreas C. Müller and Sarah Guido
Copyright © 2017 Sarah Guido, Andreas Müller. All rights reserved.
Printed in the United States of America.
Published by O’Reilly Media, Inc., 1005 Gravenstein Highway North, Sebastopol, CA 95472.
O’Reilly books may be purchased for educational, business, or sales promotional use. Online editions are
also available for most titles ( http://safaribooksonline.com). For more information, contact our corporate/
institutional sales department: 800-998-9938 or corporate@oreilly.com.
Editor: Dawn Schanafelt
Production Editor: Kristen Brown
Copyeditor: Rachel He

In [19]:
import re

def clean_text(text):
    # Fix spaced letters like "P y t h o n"
    text = re.sub(r'\b(\w)\s+(?=\w\b)', r'\1', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)

    # Fix HTML entities
    text = text.replace("&amp;", "&")

    return text

text = clean_text(text)

print("Cleaned Sample:", text[:500])

Cleaned Sample: Andreas C. Müller & Sarah Guido Introduction to Machine Learning with PythonA GUIDE FOR DATA SCIENTISTS Andreas C. Müller and Sarah Guido Introduction to Machine Learning with Python A Guide for Data Scientists Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing 978-1-449-36941-5 [LSI] Introduction to Machine Learning with Python by Andreas C. Müller and Sarah Guido Copyright © 2017 Sarah Guido, Andreas Müller. All rights reserved. Printed in the United States of Americ


In [20]:
print("Total Characters:", len(text))

Total Characters: 669026


In [21]:
!pip install -U langchain langchain-text-splitters

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Define splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,
    separators=["\n\n", "\n", ".", " ", ""]
)

# Split text
chunks = splitter.split_text(text)

# Output
print("Total Chunks:", len(chunks))
print("\nSample Chunk:\n", chunks[0][:500])



Total Chunks: 1181

Sample Chunk:
 Andreas C. Müller & Sarah Guido Introduction to Machine Learning with PythonA GUIDE FOR DATA SCIENTISTS Andreas C. Müller and Sarah Guido Introduction to Machine Learning with Python A Guide for Data Scientists Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing 978-1-449-36941-5 [LSI] Introduction to Machine Learning with Python by Andreas C. Müller and Sarah Guido Copyright © 2017 Sarah Guido, Andreas Müller. All rights reserved. Printed in the United States of Americ


In [23]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
embeddings = model.encode(chunks, normalize_embeddings=True)

In [25]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

index.add(np.array(embeddings))

In [26]:
def retrieve(query, k=5):
    query_vec = model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(query_vec, k)

    results = []
    for i in indices[0]:
        chunk = chunks[i]

        # Filter noisy chunks
        if len(chunk) > 100 and "print(" not in chunk and "In[" not in chunk:
            results.append(chunk)

    return results[:3]

In [27]:
query = "What is overfitting?"
results = retrieve(query)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n", r)


Result 1:
 . Choosing too simple a model is called underfitting. The more complex we allow our model to be, the better we will be able to predict on the training data. However, if our model becomes too complex, we start focusing too much on each individual data point in our training set, and the model will not gener‐ alize well to new data. There is a sweet spot in between that will yield the best generalization performance. This is the model we want to find. The trade-off between overfitting and underfitting is illustrated in Figure 2-1. 28 | Chapter 2: Supervised Learning Figure 2-1

Result 2:
 . The over‐ fitting can be seen on the left of Figure 2-26. Y ou can see the regions determined to belong to class 1 in the middle of all the points belonging to class 0. On the other hand, there is a small strip predicted as class 0 around the point belonging to class 0 to the very right. This is not how one would imagine the decision boundary to look, and the decision boundary focuses a lot

In [28]:
for k in [3, 5, 7]:
    print(f"\nTop {k} results:\n")
    results = retrieve("What is supervised learning?", k=k)
    for r in results:
        print("-", r[:200])


Top 3 results:

- . Given a new email, the algorithm will then produce a prediction as to whether the new email is spam. Machine learning algorithms that learn from input/output pairs are called supervised learning alg
- . xii | Preface CHAPTER 1 Introduction Machine learning is about extracting knowledge from data. It is a research field at the intersection of statistics, artificial intelligence, and computer science
- . 250 | Chapter 4: Representing Data and Engineering Features CHAPTER 5 Model Evaluation and Improvement Having discussed the fundamentals of supervised and unsupervised learning, and having explored 

Top 5 results:

- . Given a new email, the algorithm will then produce a prediction as to whether the new email is spam. Machine learning algorithms that learn from input/output pairs are called supervised learning alg
- . xii | Preface CHAPTER 1 Introduction Machine learning is about extracting knowledge from data. It is a research field at the intersection of statistic

In [29]:
def build_prompt(query, context_chunks):
    context = "\n\n".join(context_chunks)

    return f"""
You are a helpful AI assistant.

Answer the question using ONLY the given context.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {query}

Answer:
"""


In [31]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [51]:
def generate_answer(query):
    retrieved_chunks = retrieve(query)

    context = "\n".join(retrieved_chunks)
    context = context[:1200]

    prompt = f"""
You are a Machine Learning expert.

Using ONLY the context below, write a clear and complete answer in 2–3 sentences.

Context:
{context}

Question: {query}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=150,
        min_length=40,
        repetition_penalty=2.0,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [52]:
questions = [
    "What is Retrival augmented generation?",
    "Explain supervised learning",
    "What is k nearest neighbors?",
    "What are Ensemble Models?"
]

for q in questions:
    print("\nQuestion:", q)
    print("Answer:", generate_answer(q))


Question: What is Retrival augmented generation?
Answer: a bag-of-words model. Retrival augmented generation. (Retrival augmented generation) is a machine learning algorithm. The first step in the process is cross-validation.

Question: Explain supervised learning
Answer: Machine learning algorithms that learn from input/output pairs are called supervised learning algorithms because a “teacher” provides supervision to the algorithms in the form of the desired outputs for each example that they learn from.

Question: What is k nearest neighbors?
Answer: k-NN algorithm. (k-NN) algorithm is arguably the simplest machine learning algorithm. To make a prediction for a new data point, the algorithm finds the closest data points in the training dataset—its “nearest neighbors.

Question: What are Ensemble Models?
Answer: Ensembles are methods that combine multiple machine learning models to create more powerful models. The main downside of decision trees is that even with the use of pre-pruni

In [57]:
def rag_chatbot(query):
    retrieved_chunks = retrieve(query, k=5)
    answer = generate_answer(query)

    print("\n Retrieved Context:")
    for c in retrieved_chunks[:2]:
        print("-", c[:200])

    print("\n\n Answer:")
    print(answer)

In [60]:
rag_chatbot("What is Supervised Learning in machine learning?")


 Retrieved Context:
- . Given a new email, the algorithm will then produce a prediction as to whether the new email is spam. Machine learning algorithms that learn from input/output pairs are called supervised learning alg
- . xii | Preface CHAPTER 1 Introduction Machine learning is about extracting knowledge from data. It is a research field at the intersection of statistics, artificial intelligence, and computer science


 Answer:
Machine learning algorithms that learn from input/output pairs. . Given a new email, the algorithm will then produce a prediction as to whether the new email is spam. Machine learning algorithms that learn from input/output pairs are called supervised learning algorithms because a “teacher” provides supervision to the algorithms in the form of the desired outputs for each example that they learn from. While creating a dataset of inputs and outputs is often a laborious manual process, supervised learning algorithms are well understood and their performance 